<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN_DeepXDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation $p_t = \kappa p_{xx}$ for $x \in [0, \pi]$, $t \in [0, 1]$, with initial condition $p(x, 0) = \sin(x)$ and boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [2]:
import deepxde as dde
import numpy as np
from deepxde.backend import tf

Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
kappa = 1.0

In [4]:
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [5]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [6]:
def func(x):
    xx = x[:, 0]
    tt = x[:, 1]
    return (np.sin(xx) * np.exp(-tt))[:, None]

In [7]:
data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=100,
    solution=func
)

In [8]:
layer_size = [2] + [20] * 2 + [1]
activation = "tanh"
initializer = "Glorot uniform"

In [9]:
net = dde.nn.FNN(layer_size, activation, initializer)

In [10]:
def output_transform(x, y):
    x_col = x[:, 0:1] # столбец x, форма (N, 1)
    t_col = x[:, 1:2] # столбец t, форма (N, 1)
    return tf.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

net.apply_output_transform(output_transform)

In [11]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3, metrics=["l2 relative error"])

Compiling model...
'compile' took 0.051898 s



In [12]:
resampler = dde.callbacks.PDEPointResampler(period=1)

In [13]:
losshistory, train_state = model.train(iterations=1000, callbacks=[resampler], display_every=50)

Training model...

Step      Train loss    Test loss     Test metric   
0         [5.83e+00]    [5.83e+00]    [1.76e+00]    
50        [9.33e-02]    [9.33e-02]    [1.22e-01]    
100       [5.22e-02]    [5.22e-02]    [4.11e-02]    
150       [3.77e-02]    [3.77e-02]    [3.47e-02]    
200       [2.86e-02]    [2.86e-02]    [3.01e-02]    
250       [2.20e-02]    [2.20e-02]    [2.72e-02]    
300       [1.68e-02]    [1.68e-02]    [2.47e-02]    
350       [1.27e-02]    [1.27e-02]    [2.20e-02]    
400       [9.54e-03]    [9.54e-03]    [1.93e-02]    
450       [7.17e-03]    [7.17e-03]    [1.67e-02]    
500       [5.47e-03]    [5.47e-03]    [1.45e-02]    
550       [4.32e-03]    [4.32e-03]    [1.29e-02]    
600       [3.56e-03]    [3.56e-03]    [1.18e-02]    
650       [3.07e-03]    [3.07e-03]    [1.11e-02]    
700       [2.74e-03]    [2.74e-03]    [1.08e-02]    
750       [2.52e-03]    [2.52e-03]    [1.05e-02]    
800       [2.34e-03]    [2.34e-03]    [1.04e-02]    
850       [2.20e-03]    [2.

In [14]:
import plotly.express as px
import pandas as pd
import numpy as np

In [15]:
domain_x = [0, np.pi]
domain_t = [0, 1]

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
    x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
    t_plot = t_val * np.ones_like(x_plot)
    inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

    p_plot = model.predict(inputs_plot)

    p_exact = np.exp(-t_val) * np.sin(x_plot)
    df_temp = pd.DataFrame({
        'x': x_plot.flatten(),
        'PINN': p_plot.flatten(),
        'Exact': p_exact.flatten(),
        'Time': t_val
    })
    plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')

fig = px.line(df_melted, x='x', y='p', animation_frame='Time',
              color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(xaxis_title='x', yaxis_title='p(x, t)')
fig.show()

## Loss:

In [16]:
import plotly.graph_objects as go
import numpy as np

In [17]:
losses = np.array(losshistory.loss_train).flatten()
steps = np.array(losshistory.steps)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines+markers', name='Dirichlet'))
fig.update_layout(
    title='Сходимость обучения PINN (DeepXDE)',
    xaxis_title='Итерация',
    yaxis_title='Функция потерь',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

## Что будет если менять число эпох?

In [18]:
import deepxde.nn.initializers as dde_initializers

In [19]:
x = np.linspace(0, np.pi, 200)
t = np.linspace(0, 1, 100)
X, T = np.meshgrid(x, t)
inputs = np.stack([X.ravel(), T.ravel()], axis=1)
p_exact = np.exp(-T) * np.sin(X)

In [20]:
checkpoints = [1000, 2000, 3000, 4000, 5000]
results = np.zeros((5, len(checkpoints)))

In [21]:
for i, seed in enumerate(range(1, 6)):
    print(f"Обучение №{seed}")

    dde_initializers.INITIALIZER_DICT["Glorot uniform"] = tf.keras.initializers.GlorotUniform(seed=seed)

    dde.config.set_random_seed(seed)

    data = dde.data.TimePDE(geomtime, pde, [], num_domain=100, solution=func)
    net = dde.nn.FNN(layer_size, activation, initializer)
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3)

    resampler = dde.callbacks.PDEPointResampler(period=1)

    for j, step in enumerate(checkpoints):
        iterations_to_train = 1000 if j == 0 else (checkpoints[j] - checkpoints[j-1])
        model.train(iterations=iterations_to_train, callbacks=[resampler], display_every=5000)

        p_pred = model.predict(inputs).reshape(X.shape)
        results[i, j] = np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)

Обучение №1
Compiling model...
'compile' took 0.013168 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
1000      [2.52e-04]    [2.52e-04]    []  

Best model at step 1000:
  train loss: 2.52e-04
  test loss: 2.52e-04
  test metric: []

'train' took 14.171813 s

Training model...

Step      Train loss    Test loss     Test metric
1000      [2.52e-04]    [2.52e-04]    []  
2000      [4.78e-05]    [4.78e-05]    []  

Best model at step 2000:
  train loss: 4.78e-05
  test loss: 4.78e-05
  test metric: []

'train' took 12.794987 s

Training model...

Step      Train loss    Test loss     Test metric
2000      [4.78e-05]    [4.78e-05]    []  
3000      [2.30e-05]    [2.30e-05]    []  

Best model at step 3000:
  train loss: 2.30e-05
  test loss: 2.30e-05
  test metric: []

'train' took 12.222005 s

Training model...

Step      Train loss    Test loss     Test metric
3000      [2.30e-05]    [2.30e-05]    []  
4000      [1.88e-

In [22]:
mean_errors = np.mean(results, axis=0)
for step, error in zip(checkpoints, mean_errors):
    print(f"Шаг {step} -> Средняя L2 = {error:.4e}")

Шаг 1000 -> Средняя L2 = 6.6565e-03
Шаг 2000 -> Средняя L2 = 2.5075e-03
Шаг 3000 -> Средняя L2 = 1.6850e-03
Шаг 4000 -> Средняя L2 = 1.0172e-03
Шаг 5000 -> Средняя L2 = 3.0258e-03


In [30]:
fig = go.Figure()

for i in range(5):
    fig.add_trace(go.Scatter(
        x=checkpoints,
        y=results[i, :],
        mode='lines+markers',
        name=f'Обучение №{i+1}'
    ))

fig.add_trace(go.Scatter(
    x=checkpoints,
    y=mean_errors,
    mode='lines+markers',
    name='СРЕДНЕЕ ЗНАЧЕНИЕ',
    line=dict(color='black', width=4),
    marker=dict(size=10, symbol='diamond')
))

fig.update_layout(
    xaxis=dict(
        title='Количество итераций (эпох)',
        tickvals=checkpoints,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title='Относительная погрешность L2',
        exponentformat='e',
        gridcolor='lightgray'
    ),
    title=dict(
        text='Результаты 5 обучений PINN',
        x=0.5
    ),
    plot_bgcolor='white'
)

fig.show()